<a href="https://colab.research.google.com/github/LucianaNorat/eleicoes-pcd-2024-pb/blob/main/Eleitores_pcd_2024_TESTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL — Eleitores PCD 2024 PB
**Projeto:** BD Eleições PCD 2024 | **Equipe:** Luciana Norat, Mônica Vasconcelos
**Checkpoint:** C3 | Organização: Bronze → Silver → Gold

## 1. Bronze — Extração bruta das 3 fontes TSE

In [2]:
# Conecta o Google Drive ao Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile

# Extrai o ZIP direto do Drive para o Colab
with zipfile.ZipFile('/content/drive/MyDrive/Dissertacao_TSE/perfil_eleitor_secao_2024_PB.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/Dissertacao_TSE/')

print("Extração concluída!")

Extração concluída!


In [4]:
import pandas as pd

# Lê o arquivo CSV
df = pd.read_csv(
    '/content/drive/MyDrive/Dissertacao_TSE/perfil_eleitor_secao_2024_PB.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

# Mostra as primeiras linhas
df.head()

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,...,DS_IDENTIDADE_GENERO,CD_QUILOMBOLA,DS_QUILOMBOLA,CD_INTERPRETE_LIBRAS,DS_INTERPRETE_LIBRAS,TP_OBRIGATORIEDADE_VOTO,QT_ELEITORES_PERFIL,QT_ELEITORES_BIOMETRIA,QT_ELEITORES_DEFICIENCIA,QT_ELEITORES_INC_NM_SOCIAL
0,13/09/2024,15:32:22,2024,PB,21415,POCINHOS,50,15,1031,COLEGIO MUNICIPAL PADRE GALVAO,...,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,2,2,0,0
1,13/09/2024,15:32:22,2024,PB,20516,JOÃO PESSOA,64,1,1082,ESCOLA NORMAL ESTADUAL PROFESSORA MARIA DO CAR...,...,Cisgênero,2,NÃO,2,NÃO,Obrigatório,1,1,0,0
2,13/09/2024,15:32:22,2024,PB,20117,VISTA SERRANA,51,28,1104,ESCOLA MUNICIPAL DE 1º E 2º GRAUS JOSÉ GIL XAV...,...,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,1,1,0,0
3,13/09/2024,15:32:22,2024,PB,19810,CAMPINA GRANDE,17,344,1627,CEAI ANTÔNIO MARIZ,...,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,1,1,0,0
4,13/09/2024,15:32:22,2024,PB,19798,CAMALAÚ,29,175,1147,E.M.E.F. FRANCISCO CHAVES VENTURA,...,NÃO INFORMADO,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Facultativo,1,1,0,0


In [5]:
# Quantas linhas e colunas tem o arquivo?
print("Tamanho do dataset:")
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

print("\nNome de todas as colunas:")
print(df.columns.tolist())

print("\nQuantos eleitores com deficiência no total?")
print(df['QT_ELEITORES_DEFICIENCIA'].sum())

Tamanho do dataset:
Linhas: 1810719
Colunas: 31

Nome de todas as colunas:
['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'NR_LOCAL_VOTACAO', 'NM_LOCAL_VOTACAO', 'CD_GENERO', 'DS_GENERO', 'CD_ESTADO_CIVIL', 'DS_ESTADO_CIVIL', 'CD_FAIXA_ETARIA', 'DS_FAIXA_ETARIA', 'CD_GRAU_ESCOLARIDADE', 'DS_GRAU_ESCOLARIDADE', 'CD_RACA_COR', 'DS_RACA_COR', 'CD_IDENTIDADE_GENERO', 'DS_IDENTIDADE_GENERO', 'CD_QUILOMBOLA', 'DS_QUILOMBOLA', 'CD_INTERPRETE_LIBRAS', 'DS_INTERPRETE_LIBRAS', 'TP_OBRIGATORIEDADE_VOTO', 'QT_ELEITORES_PERFIL', 'QT_ELEITORES_BIOMETRIA', 'QT_ELEITORES_DEFICIENCIA', 'QT_ELEITORES_INC_NM_SOCIAL']

Quantos eleitores com deficiência no total?
24279


## 1.1 Exploração inicial (sanity check — não faz parte do pipeline formal)

In [6]:
# Total de eleitores PCD por zona eleitoral
pcd_por_zona = df.groupby('NR_ZONA')['QT_ELEITORES_DEFICIENCIA'].sum()
pcd_por_zona = pcd_por_zona.sort_values(ascending=False)

print("Top 10 zonas com mais eleitores PCD:")
print(pcd_por_zona.head(10))

Top 10 zonas com mais eleitores PCD:
NR_ZONA
77    1352
17    1041
72    1001
64     874
1      865
76     851
16     848
70     794
68     656
28     654
Name: QT_ELEITORES_DEFICIENCIA, dtype: int64


In [7]:
# Qual município corresponde a cada zona?
zona_municipio = df.groupby(['NR_ZONA', 'NM_MUNICIPIO'])['QT_ELEITORES_DEFICIENCIA'].sum()
zona_municipio = zona_municipio.sort_values(ascending=False)

print("Top 10 zonas com município:")
print(zona_municipio.head(10))

Top 10 zonas com município:
NR_ZONA  NM_MUNICIPIO  
77       JOÃO PESSOA       1352
17       CAMPINA GRANDE    1041
72       CAMPINA GRANDE     949
64       JOÃO PESSOA        874
1        JOÃO PESSOA        865
76       JOÃO PESSOA        851
70       JOÃO PESSOA        794
16       CAMPINA GRANDE     732
28       PATOS              637
61       BAYEUX             574
Name: QT_ELEITORES_DEFICIENCIA, dtype: int64


In [8]:
# Ver total de eleitores (com e sem deficiência) por município
# para comparar e identificar possível subnotificação
resumo = df.groupby('NM_MUNICIPIO').agg(
    total_eleitores=('QT_ELEITORES_PERFIL', 'sum'),
    total_pcd=('QT_ELEITORES_DEFICIENCIA', 'sum')
).reset_index()

# Calcula o percentual de PCD sobre o total
resumo['percentual_pcd'] = (resumo['total_pcd'] / resumo['total_eleitores'] * 100).round(2)

# Ordena pelo total de eleitores (maiores municípios primeiro)
resumo = resumo.sort_values('total_eleitores', ascending=False)

print("Top 15 municípios mais populosos e seus eleitores PCD:")
print(resumo.head(15).to_string(index=False))

Top 15 municípios mais populosos e seus eleitores PCD:
  NM_MUNICIPIO  total_eleitores  total_pcd  percentual_pcd
   JOÃO PESSOA           566290       4736            0.84
CAMPINA GRANDE           298888       2722            0.91
    SANTA RITA           101702        623            0.61
        BAYEUX            75831        574            0.76
         PATOS            68104        637            0.94
      CABEDELO            54561        418            0.77
    CAJAZEIRAS            47610        536            1.13
         SOUSA            47538        332            0.70
     GUARABIRA            43183        252            0.58
          SAPÉ            37342        224            0.60
     QUEIMADAS            36154        332            0.92
    MAMANGUAPE            34144        290            0.85
     SÃO BENTO            29525        109            0.37
         CONDE            27264        200            0.73
      MONTEIRO            27058        185            0.68


## 1.2 Bronze — Arquivo 2 (comparecimento/abstenção)

Extraindo os dados do arquivo de comparecimento abstenção de 2024


In [9]:
import os

caminho = '/content/drive/MyDrive/Dissertacao_TSE/'
print(os.listdir(caminho))

['perfil_eleitor_secao_2024_PB.zip', 'perfil_eleitor_secao_2024_PB.csv', 'perfil_comparecimento_abstencao_2024.zip', 'perfil_comparecimento_abstencao_2024_BRASIL.csv', 'perfil_comparecimento_abstencao_2024_PB.csv', 'leiame.pdf', 'eleitorado_local_votacao_2024_PB.csv']


In [10]:
import zipfile

with zipfile.ZipFile('/content/drive/MyDrive/Dissertacao_TSE/perfil_comparecimento_abstencao_2024.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/Dissertacao_TSE/')

print("Extração concluída!")

Extração concluída!


In [11]:
import os

caminho = '/content/drive/MyDrive/Dissertacao_TSE/'
print(os.listdir(caminho))

['perfil_eleitor_secao_2024_PB.zip', 'perfil_eleitor_secao_2024_PB.csv', 'perfil_comparecimento_abstencao_2024.zip', 'perfil_comparecimento_abstencao_2024_BRASIL.csv', 'perfil_comparecimento_abstencao_2024_PB.csv', 'leiame.pdf', 'eleitorado_local_votacao_2024_PB.csv', 'perfil_comparecimento_abstencao_2024_AC.csv', 'perfil_comparecimento_abstencao_2024_MT.csv', 'perfil_comparecimento_abstencao_2024_AL.csv', 'perfil_comparecimento_abstencao_2024_BA.csv', 'perfil_comparecimento_abstencao_2024_AM.csv', 'perfil_comparecimento_abstencao_2024_AP.csv', 'perfil_comparecimento_abstencao_2024_PA.csv', 'perfil_comparecimento_abstencao_2024_MS.csv', 'perfil_comparecimento_abstencao_2024_CE.csv', 'perfil_comparecimento_abstencao_2024_ES.csv', 'perfil_comparecimento_abstencao_2024_MA.csv', 'perfil_comparecimento_abstencao_2024_GO.csv', 'perfil_comparecimento_abstencao_2024_MG.csv', 'perfil_comparecimento_abstencao_2024_PE.csv', 'perfil_comparecimento_abstencao_2024_PI.csv', 'perfil_comparecimento_abs

In [12]:
import pandas as pd

df_comparecimento = pd.read_csv(
    '/content/drive/MyDrive/Dissertacao_TSE/perfil_comparecimento_abstencao_2024_PB.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

df_comparecimento.head()

print("Tamanho do dataset:")
print(f"Linhas: {df_comparecimento.shape[0]}")
print(f"Colunas: {df_comparecimento.shape[1]}")

print("\nNome de todas as colunas:")
print(df_comparecimento.columns.tolist())

Tamanho do dataset:
Linhas: 302946
Colunas: 43

Nome de todas as colunas:
['DT_GERACAO', 'HH_GERACAO', 'ANO_ELEICAO', 'NR_TURNO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'CD_GENERO', 'DS_GENERO', 'CD_ESTADO_CIVIL', 'DS_ESTADO_CIVIL', 'CD_FAIXA_ETARIA', 'DS_FAIXA_ETARIA', 'CD_GRAU_ESCOLARIDADE', 'DS_GRAU_ESCOLARIDADE', 'CD_COR_RACA', 'DS_COR_RACA', 'CD_QUILOMBOLA', 'DS_QUILOMBOLA', 'CD_INTERPRETE_LIBRAS', 'DS_INTERPRETE_LIBRAS', 'CD_IDENTIDADE_GENERO', 'DS_IDENTIDADE_GENERO', 'CD_IDIOMA_INDIGENA', 'DS_IDIOMA_INDIGENA', 'CD_GRUPO_INDIGENA', 'DS_GRUPO_INDIGENA', 'QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCAO', 'QT_COMPARECIMENTO_DEFICIENCIA', 'QT_ABSTENCAO_DEFICIENCIA', 'QT_COMPARECIMENTO_TTE', 'QT_ABSTENCAO_TTE', 'QT_COMPAREC_FACULTATIVO', 'QT_ABST_FACULTATIVO', 'QT_COMPAREC_OBRIGATORIO', 'QT_ABST_OBRIGATORIO', 'QT_COMPAREC_DEFIC_FACULTATIVO', 'QT_ABST_DEFIC_FACULTATIVO', 'QT_COMPAREC_DEFIC_OBRIGATORIO', 'QT_ABST_DEFIC_OBRIGATORIO']


## 1.3 Bronze — Arquivo 3 (locais de votação e acessibilidade)

In [13]:
import requests

url = "https://cdn.tse.jus.br/estatistica/sead/odsele/eleitorado_locais_votacao/eleitorado_local_votacao_2024.zip"
destino = "/content/eleitorado_local_votacao_2024.zip"

resposta = requests.get(url)
with open(destino, "wb") as f:
    f.write(resposta.content)

print("Download concluído!")

import zipfile

with zipfile.ZipFile('/content/eleitorado_local_votacao_2024.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/eleitorado_local_votacao/')

print("Extração concluída!")

import os
print(os.listdir('/content/eleitorado_local_votacao/'))

Download concluído!
Extração concluída!
['eleitorado_local_votacao_2024.csv', 'leiame.pdf']


In [14]:
import pandas as pd

# Lê o CSV nacional (ainda no disco do Colab, não no Drive)
df_local_votacao = pd.read_csv(
    '/content/eleitorado_local_votacao/eleitorado_local_votacao_2024.csv',
    sep=';',
    encoding='latin1',
    low_memory=False
)

print("Tamanho do dataset (Brasil todo):")
print(f"Linhas: {df_local_votacao.shape[0]}")
print(f"Colunas: {df_local_votacao.shape[1]}")

print("\nNome de todas as colunas:")
print(df_local_votacao.columns.tolist())

Tamanho do dataset (Brasil todo):
Linhas: 599204
Colunas: 41

Nome de todas as colunas:
['DT_GERACAO', 'HH_GERACAO', 'AA_ELEICAO', 'DT_ELEICAO', 'DS_ELEICAO', 'NR_TURNO', 'SG_UF', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'CD_TIPO_SECAO_AGREGADA', 'DS_TIPO_SECAO_AGREGADA', 'NR_SECAO_PRINCIPAL', 'NR_LOCAL_VOTACAO', 'NM_LOCAL_VOTACAO', 'CD_TIPO_LOCAL', 'DS_TIPO_LOCAL', 'DS_ENDERECO', 'NM_BAIRRO', 'NR_CEP', 'NR_TELEFONE_LOCAL', 'NR_LATITUDE', 'NR_LONGITUDE', 'CD_SITU_LOCAL_VOTACAO', 'DS_SITU_LOCAL_VOTACAO', 'CD_SITU_ZONA', 'DS_SITU_ZONA', 'CD_SITU_SECAO', 'DS_SITU_SECAO', 'CD_SITU_LOCALIDADE', 'DS_SITU_LOCALIDADE', 'CD_SITU_SECAO_ACESSIBILIDADE', 'DS_SITU_SECAO_ACESSIBILIDADE', 'QT_ELEITOR_SECAO', 'QT_ELEITOR_ELEICAO_FEDERAL', 'QT_ELEITOR_ELEICAO_ESTADUAL', 'QT_ELEITOR_ELEICAO_MUNICIPAL', 'NR_LOCAL_VOTACAO_ORIGINAL', 'NM_LOCAL_VOTACAO_ORIGINAL', 'DS_ENDERECO_LOCVT_ORIGINAL']


In [15]:
# Filtra só os registros da Paraíba
df_local_votacao_pb = df_local_votacao[df_local_votacao['SG_UF'] == 'PB'].copy()

print(f"Registros da PB: {df_local_votacao_pb.shape[0]}")

# Quais são os valores possíveis da coluna de acessibilidade?
print("\nValores únicos de acessibilidade:")
print(df_local_votacao_pb['DS_SITU_SECAO_ACESSIBILIDADE'].value_counts())

Registros da PB: 13240

Valores únicos de acessibilidade:
DS_SITU_SECAO_ACESSIBILIDADE
Sem acessibilidade    6675
Com acessibilidade    6565
Name: count, dtype: int64


In [16]:
df_local_votacao_pb.to_csv(
    '/content/drive/MyDrive/Dissertacao_TSE/eleitorado_local_votacao_2024_PB.csv',
    sep=';',
    encoding='latin1',
    index=False
)

print("Arquivo salvo no Drive!")

Arquivo salvo no Drive!


In [17]:
import os
print(os.listdir('/content/drive/MyDrive/Dissertacao_TSE/'))

['perfil_eleitor_secao_2024_PB.zip', 'perfil_eleitor_secao_2024_PB.csv', 'perfil_comparecimento_abstencao_2024.zip', 'perfil_comparecimento_abstencao_2024_BRASIL.csv', 'perfil_comparecimento_abstencao_2024_PB.csv', 'leiame.pdf', 'eleitorado_local_votacao_2024_PB.csv', 'perfil_comparecimento_abstencao_2024_AC.csv', 'perfil_comparecimento_abstencao_2024_MT.csv', 'perfil_comparecimento_abstencao_2024_AL.csv', 'perfil_comparecimento_abstencao_2024_BA.csv', 'perfil_comparecimento_abstencao_2024_AM.csv', 'perfil_comparecimento_abstencao_2024_AP.csv', 'perfil_comparecimento_abstencao_2024_PA.csv', 'perfil_comparecimento_abstencao_2024_MS.csv', 'perfil_comparecimento_abstencao_2024_CE.csv', 'perfil_comparecimento_abstencao_2024_ES.csv', 'perfil_comparecimento_abstencao_2024_MA.csv', 'perfil_comparecimento_abstencao_2024_GO.csv', 'perfil_comparecimento_abstencao_2024_MG.csv', 'perfil_comparecimento_abstencao_2024_PE.csv', 'perfil_comparecimento_abstencao_2024_PI.csv', 'perfil_comparecimento_abs

## 2. Silver — Limpeza e padronização

### 2.1 Padronizar dsGrauEscolaridade (Arquivo 1 vs Arquivo 2)

In [19]:
df['dsGrauEscolaridade'] = df['DS_GRAU_ESCOLARIDADE']
df_comparecimento['dsGrauEscolaridade'] = df_comparecimento['DS_GRAU_ESCOLARIDADE']

print("Valores em df (Arquivo 1):")
print(df['dsGrauEscolaridade'].unique())

print("\nValores em df_comparecimento (Arquivo 2):")
print(df_comparecimento['dsGrauEscolaridade'].unique())

Valores em df (Arquivo 1):
['ENSINO FUNDAMENTAL INCOMPLETO' 'SUPERIOR INCOMPLETO' 'LÊ E ESCREVE'
 'ANALFABETO' 'ENSINO MÉDIO COMPLETO' 'ENSINO MÉDIO INCOMPLETO'
 'ENSINO FUNDAMENTAL COMPLETO' 'SUPERIOR COMPLETO']

Valores em df_comparecimento (Arquivo 2):
['ENSINO MÉDIO INCOMPLETO' 'ENSINO FUNDAMENTAL COMPLETO'
 'ENSINO MÉDIO COMPLETO' 'ENSINO FUNDAMENTAL INCOMPLETO'
 'SUPERIOR INCOMPLETO' 'LÊ E ESCREVE' 'SUPERIOR COMPLETO' 'ANALFABETO']


In [20]:
# qtEletoresBiometria é opcional no modelo (nem todo perfil tem biometria cadastrada).
# Verifica quantos registros estão sem essa informação (NaN)
print("Registros sem qtEletoresBiometria:")
print(df['QT_ELEITORES_BIOMETRIA'].isna().sum())

# Verifica se existem linhas duplicadas (mesmo registro repetido)
print("\nLinhas duplicadas em df (Arquivo 1):", df.duplicated().sum())
print("Linhas duplicadas em df_comparecimento (Arquivo 2):", df_comparecimento.duplicated().sum())

Registros sem qtEletoresBiometria:
0

Linhas duplicadas em df (Arquivo 1): 2563
Linhas duplicadas em df_comparecimento (Arquivo 2): 30


In [21]:
# Remove linhas duplicadas (mantém a primeira ocorrência de cada)
linhas_antes_df = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print(f"Arquivo 1: removidas {linhas_antes_df - df.shape[0]} linhas duplicadas. Total agora: {df.shape[0]}")

linhas_antes_comp = df_comparecimento.shape[0]
df_comparecimento = df_comparecimento.drop_duplicates().reset_index(drop=True)
print(f"Arquivo 2: removidas {linhas_antes_comp - df_comparecimento.shape[0]} linhas duplicadas. Total agora: {df_comparecimento.shape[0]}")

Arquivo 1: removidas 2563 linhas duplicadas. Total agora: 1808156
Arquivo 2: removidas 30 linhas duplicadas. Total agora: 302916


In [22]:
print("Valores únicos (os primeiros 20):")
print(sorted(df['QT_ELEITORES_BIOMETRIA'].unique())[:20])

print("\nResumo estatístico:")
print(df['QT_ELEITORES_BIOMETRIA'].describe())

# Compara com o total de eleitores do grupo (QT_ELEITORES_PERFIL)
# Se biometria for sempre menor ou igual ao total, é contagem (quantidade), não sim/não
print("\nBiometria é sempre <= total de eleitores do perfil?")
print((df['QT_ELEITORES_BIOMETRIA'] <= df['QT_ELEITORES_PERFIL']).all())

Valores únicos (os primeiros 20):
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]

Resumo estatístico:
count    1.808156e+06
mean     1.688408e+00
std      1.732430e+00
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      1.020000e+02
Name: QT_ELEITORES_BIOMETRIA, dtype: float64

Biometria é sempre <= total de eleitores do perfil?
True


## 3. Gold — Construção dos documentos (modelo C2)

### 3.1 Coleção municipios

In [23]:
# Para cada município, descobrir quais zonas ele tem (vem do Arquivo 1)
# e quais turnos ele teve (vem do Arquivo 2)

zonas_por_municipio = df.groupby('CD_MUNICIPIO')['NR_ZONA'].unique().apply(lambda x: sorted(x.tolist()))
turnos_por_municipio = df_comparecimento.groupby('CD_MUNICIPIO')['NR_TURNO'].unique().apply(lambda x: sorted(x.tolist()))

# Pega nome do município e UF (são fixos, então tanto faz usar o df)
nomes_municipio = df.groupby('CD_MUNICIPIO')['NM_MUNICIPIO'].first()
uf_municipio = df.groupby('CD_MUNICIPIO')['SG_UF'].first()

# Monta a lista de documentos no formato exato da coleção municipios
documentos_municipios = []
for cd_municipio in nomes_municipio.index:
    doc = {
        "_id": int(cd_municipio),
        "nmMunicipio": nomes_municipio[cd_municipio],
        "sgUf": uf_municipio[cd_municipio],
        "turnos": [int(t) for t in turnos_por_municipio.get(cd_municipio, [1])],
        "zonas": [int(z) for z in zonas_por_municipio[cd_municipio]],
    }
    documentos_municipios.append(doc)

print(f"Total de municípios montados: {len(documentos_municipios)}")
print("\nExemplo (primeiro município):")
print(documentos_municipios[0])

Total de municípios montados: 223

Exemplo (primeiro município):
{'_id': 19003, 'nmMunicipio': 'RIACHÃO DO POÇO', 'sgUf': 'PB', 'turnos': [1], 'zonas': [4]}


### 3.2 Coleção zonas

In [24]:
# Para cada zona, descobrir quais municípios ela abrange (Arquivo 1)
municipios_por_zona = df.groupby('NR_ZONA')['CD_MUNICIPIO'].unique().apply(lambda x: sorted(int(c) for c in x))

documentos_zonas = []
for nr_zona in municipios_por_zona.index:
    doc = {
        "_id": int(nr_zona),
        "nmSede": None,  # placeholder — será enriquecido depois com o PDF do TRE-PB
        "municipios": municipios_por_zona[nr_zona],
    }
    documentos_zonas.append(doc)

print(f"Total de zonas montadas: {len(documentos_zonas)}")
print("\nExemplo (primeira zona):")
print(documentos_zonas[0])

Total de zonas montadas: 68

Exemplo (primeira zona):
{'_id': 1, 'nmSede': None, 'municipios': [20516]}


### 3.2.1 Validação cruzada com conhecimento institucional (TRE-PB)

Confere se a distribuição de municípios por zona bate com o que se sabe sobre o eleitorado da PB: cidades grandes (João Pessoa, Campina Grande, Cabedelo, Bayeux, Guarabira) costumam ocupar uma zona inteira sozinhas, enquanto municípios menores compartilham zona com vizinhos.

In [25]:
# Quantos municípios cada zona tem? (queremos ver a distribuição)
qtd_municipios_por_zona = {doc["_id"]: len(doc["municipios"]) for doc in documentos_zonas}

print("Quantas zonas têm exatamente 1 município (são grandes demais pra compartilhar):")
zonas_1_municipio = [z for z, qtd in qtd_municipios_por_zona.items() if qtd == 1]
print(len(zonas_1_municipio), "zonas:", zonas_1_municipio)

# Pra cada uma dessas zonas "sozinhas", qual é o nome do município?
print("\nNomes desses municípios:")
for nr_zona in zonas_1_municipio:
    cd_municipio = documentos_zonas[[d["_id"] for d in documentos_zonas].index(nr_zona)]["municipios"][0]
    print(f"Zona {nr_zona} -> {nomes_municipio[cd_municipio]}")

Quantas zonas têm exatamente 1 município (são grandes demais pra compartilhar):
9 zonas: [1, 10, 17, 57, 61, 64, 70, 76, 77]

Nomes desses municípios:
Zona 1 -> JOÃO PESSOA
Zona 10 -> GUARABIRA
Zona 17 -> CAMPINA GRANDE
Zona 57 -> CABEDELO
Zona 61 -> BAYEUX
Zona 64 -> JOÃO PESSOA
Zona 70 -> JOÃO PESSOA
Zona 76 -> JOÃO PESSOA
Zona 77 -> JOÃO PESSOA


In [26]:
# Descobre o código de Campina Grande
cd_campina_grande = nomes_municipio[nomes_municipio == 'CAMPINA GRANDE'].index[0]
print(f"Código de Campina Grande: {cd_campina_grande}")

# Em quais zonas o código de Campina Grande aparece (sozinho ou compartilhado)?
print("\nZonas que contêm Campina Grande:")
for doc in documentos_zonas:
    if cd_campina_grande in doc["municipios"]:
        nomes_dos_municipios = [nomes_municipio[cd] for cd in doc["municipios"]]
        print(f"Zona {doc['_id']}: {nomes_dos_municipios}")

Código de Campina Grande: 19810

Zonas que contêm Campina Grande:
Zona 16: ['CAMPINA GRANDE', 'MASSARANDUBA']
Zona 17: ['CAMPINA GRANDE']
Zona 72: ['CAMPINA GRANDE', 'SERRA REDONDA']


### 3.2.2 Enriquecimento de nmSede (PDF TRE-PB)

Dicionário construído manualmente a partir do documento oficial do TRE-PB
("Municípios abrangidos pelas Zonas Eleitorais na Paraíba"), mapeando
cada zona ao município-sede (cartório eleitoral responsável).

In [27]:
# Dicionário zona -> município sede (cartório), extraído do PDF do TRE-PB
mapa_sede = {
    1: "João Pessoa", 2: "Santa Rita", 3: "Santa Rita", 4: "Sapé",
    6: "Itabaiana", 7: "Mamanguape", 8: "Ingá", 9: "Alagoa Grande",
    10: "Guarabira", 11: "Areia", 13: "Alagoa Nova", 14: "Bananeiras",
    16: "Campina Grande", 17: "Campina Grande", 18: "Umbuzeiro", 19: "Esperança",
    20: "Araruna", 22: "Campina Grande", 23: "Soledade", 24: "Cuité",
    25: "Picuí", 26: "Santa Luzia", 27: "Taperoá", 28: "Patos",
    29: "Monteiro", 30: "Teixeira", 31: "Pombal", 32: "Piancó",
    33: "Itaporanga", 34: "Princesa Isabel", 35: "Sousa", 36: "Catolé do Rocha",
    38: "Catolé do Rocha", 40: "São José de Piranhas", 41: "Conceição", 42: "Itaporanga",
    43: "Sumé", 44: "Pedras de Fogo", 47: "Guarabira", 48: "Solânea",
    49: "Queimadas", 50: "Pocinhos", 51: "Patos", 52: "Coremas",
    55: "Rio Tinto", 56: "Juazeirinho", 57: "Cabedelo", 58: "Serra Branca",
    59: "Queimadas", 60: "Jacaraú", 61: "Bayeux", 62: "Boqueirão",
    63: "Sousa", 64: "João Pessoa", 65: "Patos", 66: "Piancó",
    67: "Remígio", 68: "Cajazeiras", 69: "São Bento", 70: "João Pessoa",
    72: "Campina Grande", 73: "Alhandra", 74: "Água Branca", 75: "Gurinhém",
    76: "João Pessoa", 77: "João Pessoa",
    # 37 e 53 ficaram sem cartório identificável no PDF — a confirmar com o TRE-PB
}

# Aplica o mapeamento nos documentos de zonas
for doc in documentos_zonas:
    doc["nmSede"] = mapa_sede.get(doc["_id"])

# Confere se ficou alguma zona sem sede (None)
zonas_sem_sede = [doc["_id"] for doc in documentos_zonas if doc["nmSede"] is None]
print(f"Zonas sem nmSede identificado: {zonas_sem_sede}")

print("\nExemplo (zona 22 — sede em Campina Grande, sem o município no array):")
print([d for d in documentos_zonas if d["_id"] == 22][0])

Zonas sem nmSede identificado: [37, 53]

Exemplo (zona 22 — sede em Campina Grande, sem o município no array):
{'_id': 22, 'nmSede': 'Campina Grande', 'municipios': [19240, 19941, 20311, 21814]}


In [28]:
mapa_sede[37] = "São João do Rio do Peixe"
mapa_sede[53] = "São João do Rio do Peixe"

# Reaplica nos documentos de zonas
for doc in documentos_zonas:
    doc["nmSede"] = mapa_sede.get(doc["_id"])

# Confere de novo se sobrou alguma zona sem sede
zonas_sem_sede = [doc["_id"] for doc in documentos_zonas if doc["nmSede"] is None]
print(f"Zonas sem nmSede identificado: {zonas_sem_sede}")

Zonas sem nmSede identificado: []
